In [1]:
# Install required libraries
!pip install flask fastapi uvicorn scikit-learn numpy pandas python-multipart


   ----------------------------------------  0/10 [werkzeug]
   ----------------------------------------  0/10 [werkzeug]
   ---- -----------------------------------  1/10 [python-multipart]
   ------------ ---------------------------  3/10 [click]
   ---------------- -----------------------  4/10 [blinker]
   ------------------------ ---------------  6/10 [uvicorn]
   ------------------------ ---------------  6/10 [uvicorn]
   ---------------------------- -----------  7/10 [starlette]
   ---------------------------- -----------  7/10 [starlette]
   -------------------------------- -------  8/10 [flask]
   -------------------------------- -------  8/10 [flask]
   ------------------------------------ ---  9/10 [fastapi]
   ------------------------------------ ---  9/10 [fastapi]
   ---------------------------------------- 10/10 [fastapi]



# For Testing APIs
- Install Postman in your system
- Run server in your local machine
- Send requests from postman to server runnning on your local machine

# Alternate route
- Run server in your local machine
- Send requests from another terminal tab in your local machine to server runnning on your local machine. For this you will need to write a script in python using requests and json library.

In [41]:
# Import necessary libraries for Flask
import flask
from flask import Flask, request, jsonify
import numpy as np
import pandas as pd
import pickle
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

In [42]:
# Create a simple ML model to deploy
def create_model():
    # Load iris dataset
    iris = load_iris()
    X = iris.data
    y = iris.target

    # Train a simple RandomForest model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X, y)

    # Save model
    with open('rf_model.pkl', 'wb') as f:
        pickle.dump(model, f)

    return model, iris.feature_names

# Create and save the model
model, feature_names = create_model()

In [3]:
# FLASK DEPLOYMENT ==============================================================


In [4]:
# Create Flask app
app = Flask(__name__)

In [43]:
# Load model for prediction
def load_model():
    with open('rf_model.pkl', 'rb') as f:
        model = pickle.load(f)
    return model

In [6]:
# Define prediction endpoint
@app.route('/predict', methods=['POST'])
def predict():
    # Get data from POST request
    data = request.get_json()

    # Check if data is in correct format
    if not data or 'features' not in data:
        return jsonify({'error': 'No valid features provided'}), 400

    # Convert data to numpy array for prediction
    try:
        features = np.array(data['features']).reshape(1, -1)
        model = load_model()
        prediction = model.predict(features).tolist()
        prediction_proba = model.predict_proba(features).tolist()

        return jsonify({
            'prediction': prediction,
            'probability': prediction_proba
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500

In [7]:
# Run Flask app
if __name__ == '__main__':
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

C:\Users\HP\miniconda3\envs\ds_learning\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [8]:
# Test our Flask API (Note: this will not work in actual execution since the Flask server needs to be running)
# This is just to demonstrate the request format
import requests
import json


In [9]:
# Example features for Iris dataset (sepal length, sepal width, petal length, petal width)
example_features = [5.1, 3.5, 1.4, 0.2]  # Example of Iris setosa

In [10]:
# The following would work if the Flask server was running
"""
response = requests.post('http://127.0.0.1:5000/predict',
                        json={'features': example_features})
print(response.json())
"""

"\nresponse = requests.post('http://127.0.0.1:5000/predict',\n                        json={'features': example_features})\nprint(response.json())\n"

In [ ]:
# FASTAPI DEPLOYMENT ===========================================================


In [44]:

# Restart the notebook runtime before running this cell if you ran the Flask app

from fastapi import FastAPI, HTTPException, Body
from pydantic import BaseModel
import uvicorn
from typing import List, Dict, Any, Optional
import numpy as np
import pickle

In [45]:
# Define data model for API input
class IrisFeatures(BaseModel):
    features: List[float]

    class ConfigDict:
        schema_extra = {
            "example": {
                "features": [5.1, 3.5, 1.4, 0.2]
            }
        }


In [46]:
# Define data model for API output
class PredictionResponse(BaseModel):
    prediction: List[int]
    probability: List[List[float]]
    feature_importance: Optional[Dict[str, float]] = None


In [47]:
# Create FastAPI app
app_fastapi = FastAPI(
    title="Iris Classifier API",
    description="API for classifying iris flowers using a RandomForest model",
    version="1.0.0"
)


In [48]:
# Load the model at startup
model = load_model()

In [49]:
@app_fastapi.post("/predict", response_model=PredictionResponse)
async def predict_iris(iris_data: IrisFeatures):
    try:
        # Convert input features to numpy array
        features = np.array(iris_data.features).reshape(1, -1)

        # Make predictions
        prediction = model.predict(features).tolist()
        prediction_proba = model.predict_proba(features).tolist()

        # Get feature importance
        feature_importance = {}
        if hasattr(model, 'feature_importances_'):
            for i, importance in enumerate(model.feature_importances_):
                feature_name = feature_names[i] if i < len(feature_names) else f"feature_{i}"
                feature_importance[feature_name] = float(importance)

        return {
            "prediction": prediction,
            "probability": prediction_proba,
            "feature_importance": feature_importance
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


In [50]:
# Health check endpoint
@app_fastapi.get("/health")
async def health_check():
    return {"status": "healthy"}


In [ ]:

# Run FastAPI app
if __name__ == "__main__":
    uvicorn.run("app:app_fastapi", host="0.0.0.0", port=8000, reload=True)


INFO:     Will watch for changes in these directories: ['D:\\AI_BootCamp\\Week 5 Day 1 - 27 July']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [16568] using StatReload


In [ ]:
# DEPLOY MODEL WITH BATCH PREDICTIONS IN FASTAPI ===============================


In [51]:
from fastapi import FastAPI, BackgroundTasks, File, UploadFile
import pandas as pd
import io
import asyncio
import time

In [52]:
# Create a new FastAPI app for batch processing
batch_app = FastAPI(title="Batch Prediction API")

In [53]:
# Queue to store batch jobs
job_queue = {}

In [54]:
# Function to process a batch job
async def process_batch_job(job_id: str, file_content: bytes):
    try:
        # Simulate processing delay
        await asyncio.sleep(2)

        # Read CSV content
        df = pd.read_csv(io.BytesIO(file_content))

        # Make predictions on batch data
        predictions = []
        for _, row in df.iterrows():
            # Extract the features from each row
            features = row.values.reshape(1, -1)
            pred = model.predict(features)[0]
            predictions.append(int(pred))

        # Update job status
        job_queue[job_id] = {
            "status": "completed",
            "predictions": predictions,
            "processed_at": time.strftime("%Y-%m-%d %H:%M:%S")
        }
    except Exception as e:
        job_queue[job_id] = {
            "status": "failed",
            "error": str(e),
            "processed_at": time.strftime("%Y-%m-%d %H:%M:%S")
        }


In [58]:
# Submit a batch job
@app_fastapi.post("/batch-predict")
async def create_batch_job(background_tasks: BackgroundTasks,
                           file: UploadFile = File(...)):
    try:
        # Generate a job ID
        job_id = f"job_{int(time.time())}"

        # Read file content
        file_content = await file.read()

        # Add job to queue with initial status
        job_queue[job_id] = {
            "status": "processing",
            "submitted_at": time.strftime("%Y-%m-%d %H:%M:%S")
        }

        # Start background task to process the job
        background_tasks.add_task(process_batch_job, job_id, file_content)

        return {"job_id": job_id, "status": "submitted"}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


In [59]:
# Check status of a batch job
@app_fastapi.get("/batch-status/{job_id}")
async def get_batch_status(job_id: str):
    if job_id not in job_queue:
        raise HTTPException(status_code=404, detail="Job not found")

    return job_queue[job_id]

In [60]:
# MODEL VERSIONING AND A/B TESTING IN FASTAPI =================================

from fastapi import Depends, Query
import random


In [ ]:
# Create a versioned API app
version_app = FastAPI(title="Versioned Model API")


In [61]:
# Simulate multiple model versions
models = {
    "v1": model,  # Our original model
    "v2": model   # In real-world, this would be a different model version
}

In [62]:
# Model registry to track model metadata
model_registry = {
    "v1": {
        "version": "1.0.0",
        "created_at": "2023-01-01",
        "accuracy": 0.95,
        "training_data": "iris_dataset_2022"
    },
    "v2": {
        "version": "2.0.0",
        "created_at": "2023-06-01",
        "accuracy": 0.97,
        "training_data": "iris_dataset_2023"
    }
}

In [63]:
# Traffic allocation for A/B testing (70% to v1, 30% to v2)
model_weights = {"v1": 0.7, "v2": 0.3}


In [64]:
# Function to select model version based on weights
def get_model_version(ab_test: bool = Query(False, description="Enable A/B testing")):
    if ab_test:
        # Perform weighted random selection
        versions = list(model_weights.keys())
        weights = list(model_weights.values())
        selected_version = random.choices(versions, weights=weights, k=1)[0]
        return selected_version
    else:
        # Default to latest version
        return "v2"


In [66]:
# Predict endpoint with versioning
@app_fastapi.post("/v-predict")
async def predict_with_version(
    iris_data: IrisFeatures,
    version: str = Query(None, description="Model version (v1 or v2)"),
    model_version: str = Depends(get_model_version)
):
    # Use specified version or the one from A/B testing
    selected_version = version if version else model_version

    if selected_version not in models:
        raise HTTPException(status_code=404, detail=f"Model version {selected_version} not found")

    selected_model = models[selected_version]

    # Convert input features to numpy array
    features = np.array(iris_data.features).reshape(1, -1)

    # Make predictions
    prediction = selected_model.predict(features).tolist()
    prediction_proba = selected_model.predict_proba(features).tolist()

    return {
        "prediction": prediction,
        "probability": prediction_proba,
        "model_version": selected_version,
        "model_info": model_registry[selected_version]
    }

In [67]:
# Get model registry information
@app_fastapi.get("/v-models")
async def get_models():
    return model_registry

In [68]:
# MODEL MONITORING IN FASTAPI =================================================

from fastapi import BackgroundTasks
import time
import uuid
from datetime import datetime

# Create monitoring API app
monitoring_app = FastAPI(title="Model Monitoring API")

In [69]:
# Storage for prediction logs
prediction_logs = []

# Model performance metrics
model_metrics = {
    "total_requests": 0,
    "successful_predictions": 0,
    "failed_predictions": 0,
    "avg_response_time": 0,
    "prediction_distribution": {0: 0, 1: 0, 2: 0},
    "last_updated": None
}

In [70]:
# Log prediction for monitoring
async def log_prediction(features, prediction, response_time, success):
    global model_metrics

    # Update request counters
    model_metrics["total_requests"] += 1

    if success:
        model_metrics["successful_predictions"] += 1

        # Update prediction distribution
        pred_class = prediction[0]
        model_metrics["prediction_distribution"][pred_class] += 1
    else:
        model_metrics["failed_predictions"] += 1

    # Update average response time
    current_avg = model_metrics["avg_response_time"]
    model_metrics["avg_response_time"] = (current_avg * (model_metrics["total_requests"] - 1) +
                                         response_time) / model_metrics["total_requests"]

    # Update timestamp
    model_metrics["last_updated"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Store detailed log
    prediction_logs.append({
        "id": str(uuid.uuid4()),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "features": features,
        "prediction": prediction,
        "response_time": response_time,
        "success": success
    })

    # Keep only last 100 logs
    if len(prediction_logs) > 100:
        prediction_logs.pop(0)


In [72]:
# Prediction endpoint with monitoring
@monitoring_app.post("/predict")
async def predict_with_monitoring(iris_data: IrisFeatures, background_tasks: BackgroundTasks):
    start_time = time.time()
    success = True
    prediction = None

    try:
        # Convert input features to numpy array
        features = np.array(iris_data.features).reshape(1, -1)

        # Make predictions
        prediction = model.predict(features).tolist()
        prediction_proba = model.predict_proba(features).tolist()

        response = {
            "prediction": prediction,
            "probability": prediction_proba
        }
    except Exception as e:
…        prediction, 
        response_time,
        success
    

    return response

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 26)

In [ ]:
# Get monitoring metrics
@monitoring_app.get("/metrics")
async def get_metrics():
    return model_metrics

In [ ]:
# Get recent prediction logs
@monitoring_app.get("/logs")
async def get_logs(limit: int = Query(10, ge=1, le=100)):
    return prediction_logs[-limit:]

In [18]:
# DEPLOY AN ENSEMBLE MODEL IN FASTAPI =========================================
from typing import Dict, List
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.datasets import load_iris
from sklearn.calibration import CalibratedClassifierCV
from fastapi import FastAPI


In [19]:
# Create ensemble models
def create_ensemble_models():
    # Load iris dataset
    iris = load_iris()
    X = iris.data
    y = iris.target

    # Train different models
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    lr_model = LogisticRegression(random_state=42)
    base_svc = SVC()
    svm_model = CalibratedClassifierCV(base_svc, ensemble=False)
    # Fit models
    rf_model.fit(X, y)
    lr_model.fit(X, y)
    svm_model.fit(X, y)

    # Save models
    ensemble = {
        "random_forest": rf_model,
        "logistic_regression": lr_model,
        "svm": svm_model
    }

    return ensemble, iris.feature_names

In [20]:
# Create ensemble
ensemble_models, _ = create_ensemble_models()

C:\Users\HP\miniconda3\envs\ds_learning\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [21]:
# Create FastAPI app for ensemble
ensemble_app = FastAPI(title="Ensemble Model API")

# Define weights for each model
model_weights = {
    "random_forest": 0.5,
    "logistic_regression": 0.3,
    "svm": 0.2
}

In [22]:
# Predict using ensemble
@ensemble_app.post("/predict")
async def predict_ensemble(iris_data: IrisFeatures):
    # Convert input features to numpy array
    features = np.array(iris_data.features).reshape(1, -1)

    # Get predictions from each model
    predictions = {}
    probabilities = {}

    for model_name, model in ensemble_models.items():
        predictions[model_name] = model.predict(features)[0]
        probabilities[model_name] = model.predict_proba(features)[0].tolist()

    # Calculate weighted ensemble prediction
    ensemble_proba = np.zeros(3)  # 3 classes for Iris

    for model_name, proba in probabilities.items():
        weight = model_weights.get(model_name, 1/len(ensemble_models))
        ensemble_proba += weight * np.array(proba)

    # Get final prediction
    final_prediction = int(np.argmax(ensemble_proba))

    return {
        "ensemble_prediction": final_prediction,
        "ensemble_probability": ensemble_proba.tolist(),
        "individual_predictions": predictions,
        "individual_probabilities": probabilities
    }

NameError: name 'IrisFeatures' is not defined

In [ ]:
# MODEL SERVING WITH CACHING IN FASTAPI =======================================

from fastapi import Depends, HTTPException, status
from fastapi.security import APIKeyHeader
import functools
import time

In [ ]:
# Create a FastAPI app with caching and authentication
cache_app = FastAPI(title="Model API with Caching")

# Simple in-memory cache
prediction_cache = {}
CACHE_TTL = 300  # Time to live: 5 minutes


In [ ]:
# API key auth
API_KEY = "your-secret-api-key"
api_key_header = APIKeyHeader(name="X-API-Key")

In [ ]:
# Verify API key
def verify_api_key(api_key: str = Depends(api_key_header)):
    if api_key != API_KEY:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid API Key"
        )
    return api_key

In [ ]:
# Cache decorator for predictions
def cache_prediction(ttl=CACHE_TTL):
    def decorator(func):
        @functools.wraps(func)
        async def wrapper(*args, **kwargs):
            # Create cache key from input features
            if 'iris_data' in kwargs:
                features_tuple = tuple(kwargs['iris_data'].features)
                cache_key = f"prediction_{features_tuple}"

                # Check if result is in cache and not expired
                if cache_key in prediction_cache:
                    result, timestamp = prediction_cache[cache_key]
                    if time.time() - timestamp < ttl:
                        return {**result, "cache_hit": True}

            # If not in cache or expired, compute result
            result = await func(*args, **kwargs)

            # Store result in cache
            if 'iris_data' in kwargs:
                prediction_cache[cache_key] = (result, time.time())

            return {**result, "cache_hit": False}
        return wrapper
    return decorator

In [ ]:
# Predict with caching
@cache_app.post("/predict")
@cache_prediction(ttl=300)
async def predict_with_cache(
    iris_data: IrisFeatures,
    api_key: str = Depends(verify_api_key)
):
    # Convert input features to numpy array
    features = np.array(iris_data.features).reshape(1, -1)

    # Make predictions (simulating slow operation)
    time.sleep(1)  # Simulate processing time
    prediction = model.predict(features).tolist()
    prediction_proba = model.predict_proba(features).tolist()

    return {
        "prediction": prediction,
        "probability": prediction_proba,
    }


In [ ]:
# Get cache statistics
@cache_app.get("/cache-info")
async def get_cache_info(api_key: str = Depends(verify_api_key)):
    current_time = time.time()
    active_cache_entries = sum(1 for _, (_, timestamp) in prediction_cache.items()
                              if current_time - timestamp < CACHE_TTL)

    return {
        "total_entries": len(prediction_cache),
        "active_entries": active_cache_entries,
        "cache_ttl": CACHE_TTL
    }

In [ ]:
# Clear cache
@cache_app.post("/clear-cache")
async def clear_cache(api_key: str = Depends(verify_api_key)):
    prediction_cache.clear()
    return {"message": "Cache cleared successfully"}

# Excercise

In [ ]:
# EXERCISE: Build a Prediction API with Rate Limiting and Performance Metrics
# ----------------------------------------------------------------------------
# In this exercise, you will:
# 1. Create a FastAPI application that serves a machine learning model
# 2. Implement rate limiting to prevent API abuse
# 3. Add performance tracking to measure prediction latency
# 4. Implement a custom logging system to track model inputs and outputs
# 5. Create a dashboard endpoint to visualize model performance

In [74]:
# SETUP: Run this cell first to prepare the exercise environment
!pip install fastapi uvicorn scikit-learn numpy pandas python-multipart

In [19]:
import numpy as np
import pandas as pd
from fastapi import FastAPI, Request, HTTPException, Depends
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import List, Dict, Any, Optional
import time
import uuid
from datetime import datetime, timedelta
import time
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
import pickle
import uvicorn

In [20]:
# Create a simple ML model for the exercise
iris = load_iris()
X = iris.data
y = iris.target
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [21]:
# Save model as a global variable for this exercise
iris_model = model
feature_names = iris.feature_names

# Define the input data model
class IrisFeatures(BaseModel):
    features: List[float]

    class ConfigDict:
        schema_extra = {
            "example": {
                "features": [5.1, 3.5, 1.4, 0.2]
            }
        }

In [22]:
# YOUR TASK: Complete the following code to build the API with rate limiting and performance tracking

# PART 1: Create the FastAPI application with rate limiting
# ---------------------------------------------------------
# Implement a rate limiter that allows each client (identified by IP address)
# to make a maximum of 5 requests per minute

In [23]:

class RateLimiter:
    def __init__(self, requests_limit: int = 5, window_seconds: int = 60):
        self.requests_limit = requests_limit
        self.window_seconds = window_seconds
        # client_id -> list of timestamps (in seconds) of their recent requests
        self.request_log: dict[str, list[float]] = defaultdict(list)

    def _prune_old_requests(self, client_id: str) -> None:
        """Remove timestamps that have fallen outside the current window."""
        now = time.time()
        cutoff = now - self.window_seconds
        self.request_log[client_id] = [
            ts for ts in self.request_log[client_id] if ts > cutoff
        ]

    def is_rate_limited(self, client_id: str) -> bool:
        self._prune_old_requests(client_id)
        return len(self.request_log[client_id]) >= self.requests_limit

    def add_request(self, client_id: str) -> None:
        self.request_log[client_id].append(time.time())

In [24]:
# Create the app and rate limiter
app = FastAPI(title="Iris Model API with Rate Limiting")
# TODO: Initialize the rate limiter

In [25]:
rate_limiter = RateLimiter(requests_limit=5, window_seconds=60)

async def check_rate_limit(request: Request):
    # request.client gives you the caller's IP/port info
    client_id = request.client.host

    if rate_limiter.is_rate_limited(client_id):
        raise HTTPException(
            status_code=status.HTTP_429_TOO_MANY_REQUESTS,
            detail=f"Rate limit exceeded. Max {rate_limiter.requests_limit} "
                   f"requests per {rate_limiter.window_seconds} seconds."
        )

    rate_limiter.add_request(client_id)

In [26]:
# PART 2: Implement performance tracking
# --------------------------------------
# Create a system to track prediction latency and model performance

import statistics

class PerformanceTracker:
    def __init__(self):
        # A plain list that grows over time — each prediction appends
        # one number (its latency in ms) to it.
        self.latencies: list[float] = []

    def record(self, latency_ms: float) -> None:
        # Just append the new measurement. Order is preserved automatically.
        self.latencies.append(latency_ms)

    def get_stats(self) -> dict:
        # Edge case: nobody has called /predict yet, so the list is empty.
        # mean()/min()/max() on an empty list would crash — guard against it.
        if not self.latencies:
            return {"count": 0, "avg_ms": 0, "min_ms": 0, "max_ms": 0}

        return {
            "count": len(self.latencies),
            "avg_ms": round(statistics.mean(self.latencies), 2),
            "min_ms": round(min(self.latencies), 2),
            "max_ms": round(max(self.latencies), 2),
        }

In [27]:
performance_metrics = {
    "request_count": 0,
    "total_latency_ms": 0.0,
    "min_latency_ms": None,   
    "max_latency_ms": None,
}

In [28]:
async def update_metrics(features, prediction, response_time):
    # response_time is expected in the same unit you pass in —
    # we'll assume milliseconds, matching what we've used so far.

    # 1. Update the running performance stats
    performance_metrics["request_count"] += 1
    performance_metrics["total_latency_ms"] += response_time
    if performance_metrics["min_latency_ms"] is None or response_time < performance_metrics["min_latency_ms"]:
        performance_metrics["min_latency_ms"] = response_time
    if performance_metrics["max_latency_ms"] is None or response_time > performance_metrics["max_latency_ms"]:
        performance_metrics["max_latency_ms"] = response_time

    # 2. Log this individual prediction (input + output + timing)
    #    This is what Part 3's logging system will read from later.
    log_entry = {
        "timestamp": time.time(),
        "features": features,
        "prediction": prediction,
        "response_time_ms": response_time,
    }
    print(log_entry)  # placeholder — swap for real logging in Part 3

In [29]:
@app.post("/predict")
async def predict(iris_data: IrisFeatures, request: Request, rate_limit: None = Depends(check_rate_limit)):
    # 1. Record start time
    start_time = time.time()

    # 2. Make prediction
    model = ml_models["rf"]
    X = np.array([[
        iris_data.sepal_length,
        iris_data.sepal_width,
        iris_data.petal_length,
        iris_data.petal_width,
    ]])
    prediction = model.predict(X)
    predicted_class = int(prediction[0])

    # 3. Calculate response time
    response_time_ms = (time.time() - start_time) * 1000

    # 4. Update metrics
    await update_metrics(iris_data.dict(), predicted_class, response_time_ms)

    # 5. Return prediction response
    return {
        "prediction": predicted_class,
        "response_time_ms": round(response_time_ms, 2),
    }

In [30]:
@app.get("/dashboard")
async def dashboard():
    count = performance_metrics["request_count"]

    # Guard against division by zero if no predictions have happened yet
    avg_latency = (
        performance_metrics["total_latency_ms"] / count if count > 0 else 0
    )

    return {
        "total_requests": count,
        "avg_response_time_ms": round(avg_latency, 2),
        "min_response_time_ms": performance_metrics["min_latency_ms"],
        "max_response_time_ms": performance_metrics["max_latency_ms"],
    }

# Run the application
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: Cannot run the event loop while another loop is running